# Models, parameters, and diagram generation

This tutorial treats a model as physics data: inspect particles and parameters, make an immutable parameter update, generate tree and loop diagrams, and inspect their graph structure.

In [ ]:
from pathlib import Path
import symbolica.community.feynkit as fk

DATA = next(path for path in (Path("data"), Path("examples/feynkit/data")) if path.exists())
model = fk.Model.from_path(DATA / "scalars_2p_3p.json")

## Particle content

Lookups are available by model name or PDG code. Spin follows the UFO convention \(2s+1\), so the scalar entries have `spin == 1`.

In [ ]:
particle_rows = [
    {
        "name": particle.name,
        "pdg": particle.pdg_code,
        "spin_2s_plus_1": particle.spin,
        "color_rep": particle.color,
        "mass_parameter": particle.mass_parameter,
        "massless": particle.is_massless,
    }
    for particle in model.particles
]
particle_rows

## Parameter cards and immutability

`ParameterCard` is mutable configuration, while `Model.with_parameter_card` returns a new model. Without an evaluator, changing an external parameter intentionally invalidates dependent internal parameters and couplings rather than leaving stale values.

In [ ]:
card = model.default_parameter_card()
card.set("lam", 2.5)
updated = model.with_parameter_card(card)

try:
    dependent_coupling = updated.coupling("SCALAR_COUPLING").value
except fk.ModelError:
    dependent_coupling = "not evaluated after the parameter update"

{
    "original_lambda": model.parameter("lam").value,
    "updated_lambda": updated.parameter("lam").value,
    "dependent_coupling_after_invalidation": dependent_coupling,
}

## Processes and inclusive loop ranges

`Process.amplitude` and `Process.cross_section` make the requested graph semantics explicit. Cross-section generation constructs cross-section graph structures; it does not numerically integrate phase space. Loop bounds are inclusive.

In [ ]:
process = (
    fk.Process.amplitude(["scalar_0"], [1000, "scalar_0"])
    .with_loop_count(0, 1)
)
{
    "kind": str(process.generation_type),
    "incoming": [str(particle) for particle in process.incoming],
    "outgoing": [str(particle) for particle in process.outgoing_alternatives[0]],
    "loops": process.loop_count,
}

`GenerationOptions` is a mutable configuration object. Methods named `add_*` and the current `set_*_filter` methods add filters; configure each filter family once to avoid duplicate-filter errors.

In [ ]:
options = fk.GenerationOptions(max_vertices=3, allow_self_loops=True)
options.add_vertex_allow(["V_3_SCALAR_000"])

generated = model.generate_diagrams(
    incoming=["scalar_0"],
    outgoing=[1000, "scalar_0"],
    loops=(0, 1),
    options=options,
)
{
    "retained": generated.report.retained_count,
    "loop_orders": sorted({diagram.loop_count for diagram in generated.diagrams}),
}

## Graph interchange and validation

Diagrams round-trip through JSON for lossless storage and through DOT for graph-tool interoperability. Validate imported diagrams against the model before using them downstream.

In [ ]:
loop_diagram = next(
    diagram
    for diagram in generated.diagrams
    if diagram.loop_count == 1
    and all(edge.source != edge.target for edge in diagram.edges)
)

from_json = fk.FeynmanDiagram.from_json(loop_diagram.to_json())
from_dot = fk.FeynmanDiagram.from_dot(loop_diagram.to_dot())
from_json.validate(model)
from_dot.validate(model)
(from_json.name, from_json.loop_count, len(from_json.edges))

## Loop-momentum bases

A basis identifies loop edges, tree edges, dependent external momenta, and the signed loop/external momentum signature carried by every edge.

In [ ]:
bases = from_json.loop_momentum_bases(limit=8)
basis = bases[0]
{
    "number_of_bases_returned": len(bases),
    "loop_edges": basis.loop_edges,
    "tree_edges": basis.tree_edges,
    "external_edges": basis.external_edges,
    "edge_momenta": [
        (edge, signature.format_momentum())
        for edge, signature in basis.edge_signatures.items()
    ],
}

## Loading UFO models

For a raw UFO model directory, install the optional dependency on Python 3.11 or newer:

```bash
pip install "symbolica[feynkit-ufo]"
```

Then configure an `fk.UfoLoader`, for example `fk.UfoLoader(restriction_name="massless").load(path)`. It returns a `LoadedModel` containing the normalized `model`, its `parameters`, and detailed loader `diagnostics`. Normalized JSON remains the reproducible, dependency-free choice for saved analyses.